# Reconstruct one quadrupole error with pyLOCO

## Learning objectives

Start from an ideal accelerator model, introduce a known quadrupole error, observe how it changes the orbit response matrix (ORM), and use pyLOCO to reconstruct the error.

**Workflow:** ideal lattice → inject quadrupole error → ORM changes → simulate BPM noise → pyLOCO fit → reconstruct the error → validate the result.

## 1. Imports and configuration

The YAML file contains user-selectable values. Python configuration structures such as `FitInitConfig` belong to `pyLOCO.config`.

In [ ]:
from copy import deepcopy
import at
import matplotlib.pyplot as plt
import numpy as np
from pyLOCO.config import FitInitConfig
from pyLOCO.pyloco import pyloco
from pyLOCO.response_matrix import response_matrix
from example_one_quad_error import (HERE, element_indices, inject_quadrupole_error, load_config, optics_beta, orm_config)

In [ ]:
config_path = HERE / 'pyloco_config.yaml'
cfg = load_config(config_path)
cfg

## 2. Load the ideal AT lattice

This lattice is the initial model supplied to LOCO. It intentionally contains none of the simulated machine error.

In [ ]:
lattice_path = config_path.parent / cfg['lattice']['file']
ideal = at.load_lattice(lattice_path, use=cfg['lattice']['use'])
ideal.disable_6d()
bpm_indices, corrector_indices, quad_indices = element_indices(ideal, cfg)
cavity_indices = np.asarray(at.get_refpts(ideal, at.elements.RFCavity), dtype=int)
print(f'{len(ideal)} lattice elements')
print(f'{len(bpm_indices)} BPMs, {len(corrector_indices)} correctors, {len(quad_indices)} fitted quadrupoles')

## 3. Inspect the selected quadrupole

The normalized quadrupole gradient $K$ has units of m⁻². Positive and negative values focus opposite transverse planes. In AT, a normal quadrupole's `K` and `PolynomB[1]` describe the same strength and must remain consistent.

In [ ]:
quad_index = int(cfg['injected_error']['quad_index'])
relative_error = float(cfg['injected_error']['relative_error'])
quad = ideal[quad_index]
s_position = float(np.asarray(at.get_s_pos(ideal, quad_index)).item())
nominal_k = float(quad.K)
print(f'Name: {quad.FamName}')
print(f'AT index: {quad_index}, s = {s_position:.6f} m')
print(f'K = {quad.K:+.9e} m^-2')
print(f'PolynomB[1] = {quad.PolynomB[1]:+.9e} m^-2')

## 4. Create the simulated machine

The ideal lattice remains LOCO's starting model. A deep copy represents the machine, and only that copy receives the configured error.

In [ ]:
machine = deepcopy(ideal)
injected_k0, injected_dk = inject_quadrupole_error(machine, quad_index, relative_error)
machine_k = float(machine[quad_index].K)
print(f'Nominal K:       {injected_k0:+.9e} m^-2')
print(f'Machine K:       {machine_k:+.9e} m^-2')
print(f'Injected ΔK:     {injected_dk:+.9e} m^-2')
print(f'Injected ΔK/K:   {100 * injected_dk / injected_k0:+.6f} %')

### Verify that the error was applied consistently

This explicit check guards against changing one AT strength representation while leaving the other stale.

In [ ]:
print('                     ideal              machine')
print(f'K             {ideal[quad_index].K:+.9e}   {machine[quad_index].K:+.9e}')
print(f'PolynomB[1]   {ideal[quad_index].PolynomB[1]:+.9e}   {machine[quad_index].PolynomB[1]:+.9e}')
assert np.isclose(machine[quad_index].K, machine[quad_index].PolynomB[1])

## 5. Calculate the ideal ORM

An ORM entry is the BPM displacement produced by the configured corrector kick. Equivalently, after normalization by the kick, $R_{ij}=\Delta x_i/\Delta\theta_j$.

In [ ]:
measurement_cfg = cfg['measurement']
kick_rad = float(measurement_cfg['corrector_kick_rad'])
rm_cfg = orm_config(bpm_indices, corrector_indices, kick_rad)
orm_ideal = response_matrix(ideal, config=rm_cfg)
print(f'Ideal ORM shape: {orm_ideal.shape}')
print(f'Ideal ORM RMS: {1e6 * np.sqrt(np.mean(orm_ideal**2)):.3f} µm per configured kick')

## 6. Calculate the erroneous-machine ORM

A quadrupole is not a corrector, but changing its focusing changes phase advance and beta functions. Corrector kicks therefore propagate differently to the BPMs, modifying the ORM.

In [ ]:
orm_machine = response_matrix(machine, config=rm_cfg)
orm_change = orm_machine - orm_ideal
print(f'RMS(machine ORM - ideal ORM): {1e6 * np.sqrt(np.mean(orm_change**2)):.6f} µm')

### Pure optics effect before adding noise

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(1e6 * orm_change, aspect='auto', cmap='RdBu_r')
ax.set(title='ORM change caused by the quadrupole error', xlabel='Corrector response column', ylabel='BPM response row')
fig.colorbar(im, ax=ax, label='Machine − ideal ORM [µm]')
plt.show()

## 7. Add reproducible BPM measurement noise

A real ORM contains BPM measurement uncertainty. Here every ORM entry receives independent Gaussian noise with the configured RMS.

In [ ]:
noise_rms_m = float(measurement_cfg['bpm_noise_rms_m'])
rng = np.random.default_rng(int(measurement_cfg['random_seed']))
noise = rng.normal(0.0, noise_rms_m, orm_machine.shape)
orm_measured = orm_machine + noise
print(f'Configured noise RMS: {1e6 * noise_rms_m:.6f} µm')
print(f'Generated noise RMS:  {1e6 * np.sqrt(np.mean(noise**2)):.6f} µm')

## 8. Prepare pyLOCO inputs

This example fits only the selected normal-quadrupole strengths. `sigma_w` expresses the same one-sigma BPM uncertainty used to generate the synthetic measurement.

In [ ]:
sigma_w = np.full(2 * len(bpm_indices), noise_rms_m)
cm_step = [[kick_rad] * len(corrector_indices)] * 2
fit_cfg = FitInitConfig(fit_list=['quads'], CMstep=cm_step, individuals=True, quads_attr='PolynomB', quads_attr_index=1)
frequency_hz = float(ideal[cavity_indices[0]].Frequency)
loco_cfg = cfg['loco']
output_dir = config_path.parent / cfg['output']['directory']
print(f'fit_list = {fit_cfg.fit_list}')
print(f'One-sigma ORM uncertainty = {1e6 * noise_rms_m:.3f} µm')

## 9. Run pyLOCO

pyLOCO changes model parameters until $R_{model}(parameters)\approx R_{measured}$. Its Jacobian describes how each ORM entry responds to a small change of each fitted quadrupole strength.

In [ ]:
result = pyloco(
    deepcopy(ideal), algorithm='lm', nIter=int(loco_cfg['nIter']),
    used_bpms_ords=bpm_indices, used_cor_ords=[corrector_indices, corrector_indices],
    quads_ords=quad_indices, skew_ords=np.array([], dtype=int), CAVords=cavity_indices,
    nHBPM=len(bpm_indices), nVBPM=len(bpm_indices),
    nHorCOR=len(corrector_indices), nVerCOR=len(corrector_indices), quads_tilt_ind=quad_indices,
    orm_measured=orm_measured, weights=sigma_w, includeDispersion=False,
    measured_eta_x=np.zeros(len(bpm_indices)), measured_eta_y=np.zeros(len(bpm_indices)),
    CMstep=cm_step, rfStep=float(measurement_cfg['rf_step_hz']), Frequency=frequency_hz,
    fit_list=['quads'], quad_individuals=True, remove_coupling_=bool(loco_cfg.get('remove_coupling', True)),
    outlier_rejection=False, apply_normalization=bool(loco_cfg.get('apply_normalization', False)),
    normalization_mode=str(loco_cfg.get('normalization_mode', 'component')),
    svd_selection_method=str(loco_cfg['svd_selection_method']), svd_threshold=float(loco_cfg['svd_threshold']),
    cut_=loco_cfg.get('cut'), show_svd_plot=False, nLMIter=int(loco_cfg['nLMIter']),
    Starting_Lambda=float(loco_cfg['Starting_Lambda']), max_lm_lambda=float(loco_cfg['max_lm_lambda']),
    scaled=bool(loco_cfg.get('scaled', True)), plot_fit_parameters=False, auto_correct_delta=True,
    fixedpathlength=False, fixedmomentum=False, fit_cfg=fit_cfg, output_dir=output_dir,
)
_, fit_dict, fitted, fitted_orm, _, chi2_history, _, _ = result

## 10. Fit convergence

A decreasing normalized chi-square means that the fitted lattice reproduces the simulated measurement more closely.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
iterations = np.arange(1, len(chi2_history) + 1)
ax.semilogy(iterations, chi2_history, 'o-')
ax.set(title='LOCO fit convergence', xlabel='LOCO iteration', ylabel='Normalized chi-square', xticks=iterations)
ax.grid(alpha=0.25)
plt.show()

## 11. Extract the reconstructed error

Both injected and fitted errors use $\Delta K=K-K_{ideal}$. The correction uses $K_{ideal}-K_{fitted}$ and therefore has the opposite sign.

In [ ]:
ideal_k = np.array([ideal[i].PolynomB[1] for i in quad_indices])
fitted_k = np.array([fitted[i].PolynomB[1] for i in quad_indices])
fitted_delta = fitted_k - ideal_k
injected_delta = np.zeros_like(fitted_delta)
injected_slot = int(np.flatnonzero(quad_indices == quad_index)[0])
injected_delta[injected_slot] = injected_dk
reconstructed_dk = float(fitted_delta[injected_slot])
reconstruction_error = reconstructed_dk - injected_dk
recovered_percent = 100 * reconstructed_dk / injected_dk
print(f'Injected ΔK/K:      {100 * injected_dk / nominal_k:+.6f} %')
print(f'Reconstructed ΔK/K: {100 * reconstructed_dk / nominal_k:+.6f} %')
print(f'Recovery:             {recovered_percent:.3f} %')
print(f'Correction ΔK:       {-reconstructed_dk:+.9e} m^-2')

## 12. Main result: injected versus reconstructed error

In [ ]:
quad_s = at.get_s_pos(ideal, quad_indices)
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.stem(quad_s, 100 * fitted_delta / ideal_k, linefmt='C0-', markerfmt='C0o', basefmt=' ', label='pyLOCO reconstructed error')
ax.scatter(quad_s[injected_slot], 100 * injected_dk / nominal_k, marker='D', s=55, color='C3', zorder=4, label='Injected error')
ax.axvline(quad_s[injected_slot], color='C3', alpha=0.25, lw=8)
ax.set(title='Injected and reconstructed quadrupole error', xlabel='Longitudinal position s [m]', ylabel='ΔK/K [%]')
ax.grid(alpha=0.25); ax.legend(); plt.show()

## 13. ORM residual before and after LOCO

The initial residual compares the ideal starting model with the measurement. The fitted residual should approach the configured BPM-noise floor.

In [ ]:
before_um = 1e6 * (orm_ideal - orm_measured)
after_um = 1e6 * (fitted_orm - orm_measured)
rms_before = float(np.sqrt(np.mean((orm_ideal - orm_measured)**2)))
rms_after = float(np.sqrt(np.mean((fitted_orm - orm_measured)**2)))
limit = np.percentile(np.abs(np.concatenate([before_um.ravel(), after_um.ravel()])), 99)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, residual, title in zip(axes, [before_um, after_um], ['Ideal model − measurement', 'Fitted model − measurement']):
    image = ax.imshow(residual, aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set(title=f'{title}\nRMS = {np.sqrt(np.mean(residual**2)):.3f} µm', xlabel='Corrector response column', ylabel='BPM response row')
fig.colorbar(image, ax=axes, label='ORM residual [µm]')
plt.show()
print(f'Improvement factor: {rms_before / rms_after:.3f}x')

## 14. Optics validation

Beta beating measures the fractional beta-function change relative to the ideal lattice. The fitted lattice should be much closer to the ideal optics than the erroneous machine.

In [ ]:
refpts = np.arange(len(ideal) + 1)
s, beta_ideal = optics_beta(ideal, refpts)
_, beta_machine = optics_beta(machine, refpts)
_, beta_fitted = optics_beta(fitted, refpts)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for plane, ax in enumerate(axes):
    ax.plot(s, 100 * (beta_machine[:, plane] - beta_ideal[:, plane]) / beta_ideal[:, plane], label='Erroneous machine', color='C3')
    ax.plot(s, 100 * (beta_fitted[:, plane] - beta_ideal[:, plane]) / beta_ideal[:, plane], label='Fitted lattice', color='C0')
    ax.set_ylabel(f'Δβ_{"xy"[plane]}/β_{"xy"[plane]} [%]'); ax.grid(alpha=0.2); ax.legend()
axes[-1].set_xlabel('Longitudinal position s [m]')
fig.suptitle('Optics validation against the ideal lattice'); plt.show()

## 15. Final numerical summary

In [ ]:
print('Single-quadrupole reconstruction summary')
print('----------------------------------------')
print(f'Quadrupole: {quad.FamName}, index {quad_index}, s = {s_position:.6f} m')
print(f'Nominal K:                 {nominal_k:+.9e} m^-2')
print(f'Injected ΔK:               {injected_dk:+.9e} m^-2 ({100 * injected_dk / nominal_k:+.6f} %)')
print(f'Reconstructed ΔK:          {reconstructed_dk:+.9e} m^-2 ({100 * reconstructed_dk / nominal_k:+.6f} %)')
print(f'Reconstruction error:      {reconstruction_error:+.9e} m^-2')
print(f'Recovered:                 {recovered_percent:.3f} %')
print(f'ORM RMS before:            {1e6 * rms_before:.6f} µm')
print(f'ORM RMS after:             {1e6 * rms_after:.6f} µm')
print(f'ORM improvement factor:    {rms_before / rms_after:.3f}x')

## Takeaway

- A quadrupole strength error changes the lattice optics.
- The optics change modifies the orbit response matrix.
- LOCO uses the ORM difference and its Jacobian to infer model parameters.
- pyLOCO reconstructed the deliberately introduced quadrupole error.

### Questions to explore

1. What happens if the injected error is reduced from 5% to 1%?
2. What happens when the BPM noise is increased?
3. Why does a quadrupole strength error modify the ORM?
4. Why is the correction sign opposite to the fitted error?